# Distributed Cluster Job Manager — Demo

Run this notebook on **FABRIC JupyterHub** (`jupyter.fabric-testbed.net`).

All logic lives in the `.py` modules. This notebook only:
1. Checks FABlib config
2. Provisions the cluster
3. Calls demo methods
4. Launches the UI
5. Tears down the slice

---
## Cell 1 — FABlib config check

In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

project_id = "f5b4fc7d-978f-45be-9530-b38db8ef5046"

fablib = fablib_manager(project_id=project_id)
fablib.show_config()

User: fozdaban@hawk.illinoistech.edu bastion key is valid!
Configuration is valid


Version,1.9.3
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
Log Level,INFO
Log File,/tmp/fablib/fablib.log
Data directory,/tmp/fablib
Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Bastion Host,bastion.fabric-testbed.net


Version,1.9.3
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
Log Level,INFO
Log File,/tmp/fablib/fablib.log
Data directory,/tmp/fablib
Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Bastion Host,bastion.fabric-testbed.net


---
## Cell 2 — Provision cluster

Creates **3 FABRIC VMs** (2 cores, because FABRIC only allows that) so the scheduler's
quota decisions are non-trivial. Takes ~2-3 minutes.

> If the kernel restarted but the slice is still alive, replace
> `system.provision(fablib)` with `system.reconnect(fablib)`.

In [2]:
from system import ClusterSystem

system = ClusterSystem()
fabric_slice = system.provision(fablib, site="GPN")

fabric_slice.show()
fabric_slice.list_nodes()

[FABRIC] Slice 'cluster-job-manager' already exists, reconnecting...
[FABRIC] Re-attached to existing slice 'cluster-job-manager'
[FABRIC]   worker-large at 2610:e0:a04c:fab2:f816:3eff:fe83:7ce9
[FABRIC]   worker-medium at 2610:e0:a04c:fab2:f816:3eff:fe6f:fee3
[FABRIC]   worker-small at 2610:e0:a04c:fab2:f816:3eff:fe86:cf2c


ID,3fc0f91e-0601-4093-8b50-393acb3eb651
Name,cluster-job-manager
Lease Expiration (UTC),2026-04-20 06:07:51 +0000
Lease Start (UTC),2026-04-19 06:07:51 +0000
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
State,StableOK
Email,fozdaban@hawk.illinoistech.edu
UserId,900c764b-4b5d-4c7d-bee9-cd4f93d6a86c


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
396dba48-94b3-4a36-9174-b81a67b7586b,worker-large,4,16,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe83:7ce9,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe83:7ce9,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
1d933a06-8ccb-4566-8425-ff21cd899f17,worker-medium,2,8,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe6f:fee3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe6f:fee3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e55fb929-db33-4c04-824f-d8d12803b17b,worker-small,2,4,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe86:cf2c,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe86:cf2c,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
396dba48-94b3-4a36-9174-b81a67b7586b,worker-large,4,16,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe83:7ce9,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe83:7ce9,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
1d933a06-8ccb-4566-8425-ff21cd899f17,worker-medium,2,8,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe6f:fee3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe6f:fee3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e55fb929-db33-4c04-824f-d8d12803b17b,worker-small,2,4,100,default_ubuntu_22,qcow2,gpn-w5.fabric-testbed.net,GPN,ubuntu,2610:e0:a04c:fab2:f816:3eff:fe86:cf2c,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2610:e0:a04c:fab2:f816:3eff:fe86:cf2c,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


---
## Cell 3 — Inspect cluster node specs

In [3]:
system.show_cluster()

Node                  Cores   Used   RAM MB   RAM Used   Util%  Tasks Status      
--------------------------------------------------------------------------------
worker-large              2      0     8192          0     0.0      0 idle        
worker-medium             2      0     6144          0     0.0      0 idle        
worker-small              2      0     4096          0     0.0      0 idle        


---
## Demo A: Quota-based scheduling + distributed execution

Three jobs with different resource demands are submitted simultaneously.
The scheduler computes **weighted fair-share quotas** across all jobs,
decomposes large jobs into chunks that fill their quota across multiple nodes,
and runs all chunks in parallel via SSH.

Wall-clock time proves real distribution: `heavy_job` runs 3 chunks on
3 VMs simultaneously, finishing in ~20s instead of ~60s sequential.

In [4]:
system.demo_quota_and_distribution()

DEMO A — Quota-based scheduling + distributed execution
[SUBMITTED] Job(60b64f3e, heavy_job, state=queued)
[SUBMITTED] Job(3198cddd, medium_job, state=queued)
[SUBMITTED] Job(a01404d2, light_job, state=queued)

Weighted fair-share quotas (computed over all jobs together):
Cluster total: 6 cores, 18432 MB RAM
Job                    Demanded  Quota Cores   Quota RAM MB   Share%
----------------------------------------------------------------------
heavy_job                     6            3           9216    50.0%
medium_job                    3            1           5820    16.7%
light_job                     1            1            970    16.7%

Scheduling all jobs...
[SCHED] Quota for heavy_job: {'cores': 3, 'ram_mb': 9216}
[SCHED] Decomposed heavy_job into 3 sub-tasks
[SCHED] Quota for medium_job: {'cores': 1, 'ram_mb': 5820}
[SCHED] Decomposed medium_job into 2 sub-tasks
[SCHED] Quota for light_job: {'cores': 1, 'ram_mb': 970}
[SCHED] Decomposed light_job into 1 sub-tasks
[SCHED

{'heavy': 'completed',
 'medium': 'completed',
 'light': 'completed',
 'wall_clock_s': 20.9}

---
## Demo B — Timeout detection

A job with `time_limit=15s` runs `sleep 999`. The monitor polls every 5s,
detects the exceeded deadline, and marks the job failed with a timeout error.

In [5]:
system.demo_timeout(poll_interval=5, polls=8)

DEMO B — Timeout error detection
[SUBMITTED] Job(e39f4803, timeout_job, state=queued)
[SUBMITTED] Job(07da8708, ok_job, state=queued)
[SCHED] Quota for timeout_job: {'cores': 3, 'ram_mb': 9216}
[SCHED] Decomposed timeout_job into 1 sub-tasks
[SCHED] Quota for ok_job: {'cores': 3, 'ram_mb': 9216}
[SCHED] Decomposed ok_job into 1 sub-tasks
[SCHED] timeout_job running on: ['worker-large']
[SCHED] ok_job running on: ['worker-medium']
[EXEC] SSH subtask e39f4803-0 on worker-large
[EXEC] SSH subtask 07da8708-0 on worker-medium

Polling monitor every 5s for timeout detection...
[EXEC] 07da8708-0 finished in 5.853s (rc=0)
[SCHED] Job ok_job completed successfully
  Poll  1: timed_out=[], failed_nodes=[]
[02:11:41] Job timeout_job timed out (16s > 15s)
  Poll  2: timed_out=['e39f4803'], failed_nodes=[]
           ↳ Timeout caught for job(s): ['e39f4803']

Final job states:
ID         Name                 State         Chunks Nodes                          Error
---------------------------------

---
## Demo C — Node failure + task rescheduling

Jobs run across all nodes. After 10s, `worker-small` is marked unresponsive.
The monitor reschedules its tasks to healthy nodes and continues execution.

In [6]:
system.demo_node_failure(fail_node="worker-small", failure_delay=10)

DEMO C — Node failure detection + task rescheduling
[SUBMITTED] Job(650913e8, node_fail_job_0, state=queued)
[SUBMITTED] Job(960d44d6, node_fail_job_1, state=queued)
[SUBMITTED] Job(9c1b2b46, node_fail_job_2, state=queued)
[SCHED] Quota for node_fail_job_0: {'cores': 2, 'ram_mb': 6144}
[SCHED] Decomposed node_fail_job_0 into 1 sub-tasks
[SCHED] Quota for node_fail_job_1: {'cores': 2, 'ram_mb': 6144}
[SCHED] Decomposed node_fail_job_1 into 1 sub-tasks
[SCHED] Quota for node_fail_job_2: {'cores': 2, 'ram_mb': 6144}
[SCHED] Decomposed node_fail_job_2 into 1 sub-tasks
[SCHED] node_fail_job_0 running on: ['worker-large']
[SCHED] node_fail_job_1 running on: ['worker-medium']
[SCHED] node_fail_job_2 running on: ['worker-small']

Initial assignment:
  node_fail_job_0 → worker-large
  node_fail_job_1 → worker-medium
  node_fail_job_2 → worker-small
[EXEC] SSH subtask 650913e8-0 on worker-large
[EXEC] SSH subtask 960d44d6-0 on worker-medium
[EXEC] SSH subtask 9c1b2b46-0 on worker-small

Jobs run

---
## Widget UI

Interactive interface: submit jobs, view quotas, monitor cluster, inspect jobs.

In [7]:
system.show_ui()

[UI] Background polling started (every 5s)


---
## Teardown

In [8]:
#system.teardown(fablib)
#print("Done — all FABRIC VMs released.")

[EXEC] SSH subtask fd65315a-0 on worker-large
[EXEC] SSH subtask ba4a451a-0 on worker-medium
[EXEC] SSH subtask ba4a451a-1 on worker-small
[EXEC] SSH subtask ba4a451a-2 on worker-large
[EXEC] SSH subtask 16900994-0 on worker-medium
[EXEC] SSH subtask 22ed6bff-0 on worker-small
[EXEC] ba4a451a-0 finished in 30.777s (rc=0)
[EXEC] SSH subtask 22ed6bff-1 on worker-medium
[EXEC] ba4a451a-2 finished in 30.781s (rc=0)
[EXEC] SSH subtask 22ed6bff-2 on worker-large
[EXEC] ba4a451a-1 finished in 31.2s (rc=0)
[SCHED] Job cpu_stress_test completed successfully
[EXEC] SSH subtask d9f67827-0 on worker-small
[EXEC] 16900994-0 finished in 31.401s (rc=0)
[SCHED] Job memory_alloc_test completed successfully
[EXEC] d9f67827-0 finished in 5.808s (rc=0)
[SCHED] Job simple_sleep completed successfully
[EXEC] fd65315a-0 finished in 40.633s (rc=0)
[SCHED] Job sleep_test completed successfully
